# Commercial Intelligence & Profitability Analytics
## 01 — Auditoría y calidad de datos

**Empresa:** Distribuidora Cordillera (datos ficticios)  
**Tecnologías:** SQLite, SQL avanzado, Jupyter

### Objetivo
Evaluar la estructura, integridad y calidad de los datos antes de desarrollar los indicadores comerciales y de rentabilidad.

### Alcance
- Revisión de tablas y relaciones.
- Validación de identificadores.
- Detección de valores ausentes e inconsistencias.
- Identificación de posibles riesgos para el análisis de Business Intelligence.

In [4]:
from pathlib import Path

print("Entorno Jupyter funcionando")
print("Directorio de trabajo:", Path.cwd())

Entorno Jupyter funcionando
Directorio de trabajo: /Users/bastianseura/Desktop/Bastian_BI/cordillera-commercial-intelligence./notebooks


In [5]:
from pathlib import Path

db_path = Path("../data/distribuidora_cordillera.sqlite").resolve()

print("Base encontrada:", db_path.is_file())
print("Tamaño:", round(db_path.stat().st_size / 1024**2, 2), "MB"
      if db_path.is_file() else "No disponible")

Base encontrada: True
Tamaño: 10.18 MB


In [6]:
%load_ext sql
%sql sqlite:///../data/distribuidora_cordillera.sqlite

Connecting to 'sqlite:///../data/distribuidora_cordillera.sqlite'

## 1. Exploración de la base de datos

### 1.1 Inventario de tablas

**Objetivo:** identificar las tablas disponibles y
comprender su estructura general.

Esta exploración establece el punto de partida
para evaluar la integridad y calidad de los datos.

In [7]:
%%sql
SELECT
    name AS nombre_tabla,
    type AS tipo_objeto
FROM sqlite_master
WHERE type = 'table'
    AND name NOT LIKE 'sqlite_%'
ORDER BY name;

Running query in 'sqlite:///../data/distribuidora_cordillera.sqlite'

nombre_tabla,tipo_objeto
calendario,table
categorias,table
clientes,table
compras,table
detalle_compras,table
detalle_pedidos,table
devoluciones,table
incidencias_calidad,table
inventario,table
metadatos,table


#### Resultados del inventario

La exploración del catálogo de SQLite permitió identificar
17 tablas en la base de datos.

La estructura disponible contempla información comercial,
clientes, productos, compras, proveedores, inventario,
devoluciones, pagos y registros operativos.

Para la primera etapa de auditoría se priorizarán cuatro
tablas fundamentales:

- `clientes`: información de la cartera comercial.
- `pedidos`: registros de operaciones comerciales.
- `detalle_pedidos`: detalle de productos por pedido.
- `productos`: catálogo de productos.

Las tablas restantes serán incorporadas progresivamente
según los requerimientos del análisis.

**Alcance inicial:** validar la integridad de los
identificadores y las relaciones entre estas cuatro tablas,
antes de construir los indicadores de negocio.

### 1.2 Auditoría de identificadores

#### Objetivo

Evaluar la unicidad y completitud de los identificadores
principales de las tablas seleccionadas.

Esta validación busca detectar registros con identificadores
duplicados o ausentes que puedan comprometer la integridad
del modelo analítico.

#### Metodología

Para cada tabla se calcularán tres indicadores:

1. **Total de registros:** cantidad de filas disponibles.
2. **Identificadores distintos:** cantidad de valores únicos,
   excluyendo valores nulos.
3. **Identificadores nulos:** registros que no presentan
   un identificador informado.

Se considerará que una columna supera estos controles
cuando:

- El total de registros coincida con los identificadores
  distintos.
- No existan identificadores nulos.

Esta comprobación evalúa la integridad individual de los
identificadores, pero no garantiza todavía la integridad
referencial entre tablas.

In [8]:
%%sql

SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT cliente_id) AS ids_distintos,
    SUM(
        CASE
            WHEN cliente_id IS NULL THEN 1
            ELSE 0
        END
    ) AS ids_nulos
FROM clientes;

Running query in 'sqlite:///../data/distribuidora_cordillera.sqlite'

total_registros,ids_distintos,ids_nulos
600,600,0


In [9]:
%%sql

SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT pedido_id) AS ids_distintos,
    SUM(
        CASE
            WHEN pedido_id IS NULL THEN 1
            ELSE 0
        END
    ) AS ids_nulos
FROM pedidos;

Running query in 'sqlite:///../data/distribuidora_cordillera.sqlite'

total_registros,ids_distintos,ids_nulos
15507,15507,0


In [10]:
%%sql

SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT detalle_id) AS ids_distintos,
    SUM(
        CASE
            WHEN detalle_id IS NULL THEN 1
            ELSE 0
        END
    ) AS ids_nulos
FROM detalle_pedidos;

Running query in 'sqlite:///../data/distribuidora_cordillera.sqlite'

total_registros,ids_distintos,ids_nulos
61647,61647,0


In [11]:
%%sql

SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT producto_id) AS ids_distintos,
    SUM(
        CASE
            WHEN producto_id IS NULL THEN 1
            ELSE 0
        END
    ) AS ids_nulos
FROM productos;

Running query in 'sqlite:///../data/distribuidora_cordillera.sqlite'

total_registros,ids_distintos,ids_nulos
180,180,0



#### 1.2.1 Resultados de la auditoría

Se evaluó la integridad de los identificadores principales en las cuatro tablas seleccionadas.

| Tabla | Registros | IDs distintos | IDs nulos | Estado |
|---|---:|---:|---:|---|
| `clientes` | 600 | 600 | 0 | Conforme |
| `pedidos` | 15.507 | 15.507 | 0 | Conforme |
| `detalle_pedidos` | 61.647 | 61.647 | 0 | Conforme |
| `productos` | 180 | 180 | 0 | Conforme |

#### Hallazgos

No se detectaron identificadores duplicados ni valores nulos en las columnas evaluadas.

En las cuatro tablas, la cantidad de registros coincide con la cantidad de identificadores distintos, lo que permite validar su unicidad en los datos examinados.

#### Conclusión

Las cuatro tablas superan los controles iniciales de unicidad y completitud de identificadores.

Sin embargo, estos resultados no permiten asegurar todavía la integridad referencial entre las entidades. Para ello, será necesario comprobar que las claves utilizadas en las relaciones correspondan a registros existentes en sus respectivas tablas de referencia.



### 1.3 Auditoría de integridad referencial

#### Objetivo

Verificar que las relaciones entre clientes, pedidos, detalles y productos sean consistentes, identificando posibles registros huérfanos.

Un registro huérfano corresponde a una fila que contiene una referencia a otra entidad cuyo registro asociado no existe en la tabla correspondiente.

#### Metodología

Se evaluarán las siguientes relaciones:

| Tabla de origen | Clave de referencia | Tabla de destino | Clave |
|---|---|---|---|
| `pedidos` | `cliente_id` | `clientes` | `cliente_id` |
| `detalle_pedidos` | `pedido_id` | `pedidos` | `pedido_id` |
| `detalle_pedidos` | `producto_id` | `productos` | `producto_id` |

Se utilizarán consultas SQL para identificar referencias sin coincidencia en las tablas correspondientes.

Los resultados permitirán determinar si existen inconsistencias capaces de afectar los análisis comerciales, la trazabilidad de pedidos y el cálculo de indicadores.


In [12]:
%%sql

SELECT
    COUNT(*) AS pedidos_sin_cliente
FROM pedidos AS p
    LEFT JOIN clientes AS c 
    ON p.cliente_id = c.cliente_id
WHERE
    c.cliente_id IS NULL
; 


Running query in 'sqlite:///../data/distribuidora_cordillera.sqlite'

pedidos_sin_cliente
0


In [13]:
%%sql

SELECT
    COUNT(*) AS detalles_sin_pedido
FROM detalle_pedidos AS dp
    LEFT JOIN pedidos AS p
    ON dp.pedido_id = p.pedido_id
WHERE
    p.pedido_id IS NULL
;


Running query in 'sqlite:///../data/distribuidora_cordillera.sqlite'

detalles_sin_pedido
0


In [14]:
%%sql

SELECT
    COUNT(*) AS detalles_sin_producto
FROM detalle_pedidos AS dp
    LEFT JOIN productos AS pr
    ON dp.producto_id = pr.producto_id
WHERE
    pr.producto_id IS NULL
;


Running query in 'sqlite:///../data/distribuidora_cordillera.sqlite'

detalles_sin_producto
0



#### 1.3.1 Resultados de integridad referencial

Se verificaron las relaciones entre las cuatro tablas principales del modelo comercial mediante consultas SQL basadas en `LEFT JOIN` y `IS NULL`.

| Control | Relación evaluada | Registros huérfanos | Estado |
|---|---|---:|---|
| A | `pedidos` → `clientes` | 0 | Conforme |
| B | `detalle_pedidos` → `pedidos` | 0 | Conforme |
| C | `detalle_pedidos` → `productos` | 0 | Conforme |

#### Hallazgos

No se detectaron registros huérfanos en ninguna de las tres relaciones evaluadas.

Todos los pedidos registrados presentan una referencia válida hacia la tabla de clientes. Asimismo, las líneas de detalle examinadas están asociadas a pedidos y productos existentes.

#### Conclusión

Las relaciones evaluadas superan los controles iniciales de integridad referencial.

Estos resultados respaldan el uso de las cuatro tablas para construir análisis comerciales mediante operaciones JOIN, sin detectar pérdidas de registros asociadas a referencias inexistentes.

**Alcance:** la validación corresponde a las relaciones seleccionadas y al estado de la base de datos durante la ejecución. No constituye una auditoría integral de todas las entidades del sistema.



### 1.4 Auditoría de calidad y completitud

#### Objetivo

Evaluar la completitud y consistencia de los atributos relevantes para el análisis comercial y operativo.

Se identificarán valores ausentes, registros potencialmente anómalos y condiciones que podrían comprometer el cálculo de indicadores de Business Intelligence.

#### Metodología

La revisión se organizará en tres dimensiones:

1. **Completitud:** presencia de valores NULL en campos relevantes.
2. **Validez:** identificación de valores fuera de rangos esperados o categorías no reconocidas.
3. **Consistencia:** verificación de relaciones lógicas entre atributos, especialmente fechas e importes.

Los hallazgos se clasificarán según su posible impacto analítico.

La ausencia de un valor no se considerará automáticamente un error; su interpretación dependerá del contexto y de las reglas del negocio.


In [15]:

%%sql

SELECT
    COUNT(*) AS total_pedidos,
    SUM(CASE WHEN pedido_id IS NULL THEN 1 ELSE 0 END) AS nulos_pedido_id,
    SUM(CASE WHEN cliente_id IS NULL THEN 1 ELSE 0 END) AS nulos_cliente_id,
    SUM(CASE WHEN fecha_pedido IS NULL THEN 1 ELSE 0 END) AS nulos_fecha_pedido,
    SUM(CASE WHEN fecha_entrega IS NULL THEN 1 ELSE 0 END) AS nulos_fecha_entrega,
    SUM(CASE WHEN fecha_vencimiento IS NULL THEN 1 ELSE 0 END) AS nulos_fecha_vencimiento,
    SUM(CASE WHEN estado IS NULL THEN 1 ELSE 0 END) AS nulos_estado
FROM pedidos;

Running query in 'sqlite:///../data/distribuidora_cordillera.sqlite'

total_pedidos,nulos_pedido_id,nulos_cliente_id,nulos_fecha_pedido,nulos_fecha_entrega,nulos_fecha_vencimiento,nulos_estado
15507,0,0,0,773,773,0



#### 1.4.1 Evaluación de completitud — Pedidos

Se analizaron los 15.507 registros de la tabla `pedidos`, evaluando la presencia de valores nulos en seis atributos relevantes.

| Campo | Valores NULL |
|---|---:|
| `pedido_id` | 0 |
| `cliente_id` | 0 |
| `fecha_pedido` | 0 |
| `fecha_entrega` | 773 |
| `fecha_vencimiento` | 773 |
| `estado` | 0 |

#### Hallazgos preliminares

No se identificaron valores ausentes en los identificadores, la fecha de creación ni el estado comercial de los pedidos.

Se detectaron 773 registros con fecha de entrega ausente y 773 con fecha de vencimiento ausente.

La cantidad de valores nulos en ambos campos coincide con el total histórico de pedidos pendientes y cancelados (75 y 698, respectivamente).

Esta coincidencia sugiere una posible relación entre la ausencia de fechas y el estado comercial del pedido. Sin embargo, todavía no permite confirmar que correspondan a los mismos registros.

**Estado del hallazgo:** pendiente de validación mediante un análisis de completitud por estado comercial.


In [16]:
%%sql

SELECT
    estado,
    COUNT(*) AS total_pedidos,
    SUM(CASE WHEN fecha_entrega IS NULL THEN 1 ELSE 0 END) AS sin_fecha_entrega,
    SUM(CASE WHEN fecha_vencimiento IS NULL THEN 1 ELSE 0 END) AS sin_fecha_vencimiento
FROM pedidos
GROUP BY estado;

Running query in 'sqlite:///../data/distribuidora_cordillera.sqlite'

estado,total_pedidos,sin_fecha_entrega,sin_fecha_vencimiento
cancelado,698,698,698
entregado,14734,0,0
pendiente,75,75,75



#### 1.4.2 Análisis de completitud según estado comercial

Con el objetivo de investigar los 773 valores nulos identificados en `fecha_entrega` y `fecha_vencimiento`, se analizó su distribución según el estado de los pedidos.

| Estado | Total pedidos | Sin fecha de entrega | Sin fecha de vencimiento |
|---|---:|---:|---:|
| entregado | 14.734 | 0 | 0 |
| pendiente | 75 | 75 | 75 |
| cancelado | 698 | 698 | 698 |
| **Total** | **15.507** | **773** | **773** |

#### Hallazgos

La ausencia de fechas presenta una correspondencia completa con el estado comercial:

- Todos los pedidos entregados presentan ambas fechas informadas.
- Los pedidos pendientes y cancelados concentran la totalidad de los valores nulos.
- No se detectaron excepciones a este patrón en los registros analizados.

#### Interpretación

La ausencia de `fecha_entrega` resulta coherente con pedidos que no han sido entregados.

La ausencia de `fecha_vencimiento` presenta el mismo patrón, aunque su justificación depende de las reglas operativas con las que se genera dicho atributo.

Por lo tanto, los valores ausentes identificados no constituyen por sí solos evidencia suficiente de errores de calidad.

#### Conclusión

Se identificó una relación consistente entre el estado comercial y la completitud de los campos de fecha evaluados.

Para el futuro análisis de tiempos de entrega, se considerarán únicamente los pedidos entregados con fechas válidas.

**Estado del control:** patrón de completitud validado. La interpretación operativa de `fecha_vencimiento` permanece sujeta a confirmación de las reglas de negocio.



### 1.4.3 Validación de consistencia temporal

#### Objetivo

Evaluar la validez y coherencia cronológica de las fechas asociadas a los pedidos entregados.

#### Metodología

Se comprobarán tres condiciones:

1. Que `fecha_pedido` pueda interpretarse como fecha.
2. Que `fecha_entrega` pueda interpretarse como fecha.
3. Que la fecha de entrega no sea anterior a la fecha de creación del pedido.

Estas validaciones buscan identificar registros capaces de distorsionar los indicadores de tiempos de entrega y desempeño logístico.


In [17]:
%%sql

SELECT
    COUNT(*) AS total_entregados,
    SUM(CASE WHEN JULIANDAY(fecha_pedido) IS NULL THEN 1 ELSE 0 END) AS fecha_pedido_invalidas,
    SUM(CASE WHEN JULIANDAY(fecha_entrega) IS NULL THEN 1 ELSE 0 END) AS fecha_entrega_invalidas,
    SUM(CASE WHEN JULIANDAY(fecha_entrega) < JULIANDAY(fecha_pedido) THEN 1 ELSE 0 END) AS entregas_anticipadas
FROM pedidos
WHERE estado = 'entregado';

Running query in 'sqlite:///../data/distribuidora_cordillera.sqlite'

total_entregados,fecha_pedido_invalidas,fecha_entrega_invalidas,entregas_anticipadas
14734,0,0,0



#### 1.4.3.1 Resultados de consistencia temporal

Se evaluaron los 14.734 pedidos con estado `entregado`, utilizando la función `JULIANDAY()` de SQLite para comprobar la interpretación de fechas y su coherencia cronológica.

| Indicador | Registros detectados |
|---|---:|
| Pedidos entregados evaluados | 14.734 |
| Fechas de pedido no interpretables | 0 |
| Fechas de entrega no interpretables | 0 |
| Entregas anteriores al pedido | 0 |

#### Hallazgos

No se identificaron fechas no interpretables por SQLite ni registros cuya fecha de entrega sea anterior a la fecha de creación del pedido.

La totalidad de los pedidos entregados supera los controles temporales aplicados.

#### Conclusión

Los resultados respaldan la utilización de los campos `fecha_pedido` y `fecha_entrega` para construir indicadores de tiempos de entrega.

No obstante, esta validación no evalúa la razonabilidad de las duraciones ni el cumplimiento de plazos logísticos.

**Estado del control:** conforme, sin anomalías detectadas.



### 1.4.4 Validación de cantidades e importes

#### Objetivo

Evaluar la completitud y validez de los atributos cuantitativos de `detalle_pedidos`, con el propósito de identificar registros que puedan distorsionar los indicadores comerciales y de rentabilidad.

#### Metodología

Se examinarán las siguientes dimensiones:

1. **Completitud:** presencia de valores nulos en cantidades, precios y costos.
2. **Validez:** identificación de cantidades no positivas e importes potencialmente anómalos.
3. **Consistencia:** revisión de las relaciones entre cantidades, precios, descuentos e importes.

Los resultados se interpretarán considerando las reglas comerciales y las características de los productos.

La detección de valores atípicos no implicará automáticamente que los registros sean incorrectos.


In [19]:
%%sql

SELECT
    COUNT(*) AS total_lineas,
    SUM(CASE WHEN cantidad IS NULL THEN 1 ELSE 0 END) AS cantidades_nulas,
    SUM(CASE WHEN cantidad <= 0 THEN 1 ELSE 0 END) AS cantidades_no_positivas,
    SUM(CASE WHEN neto_unitario_clp IS NULL THEN 1 ELSE 0 END) AS precios_unitarios_nulos,
    SUM(CASE WHEN neto_unitario_clp < 0 THEN 1 ELSE 0 END) AS precios_unitarios_negativos,
    SUM(CASE WHEN costo_unitario_clp IS NULL THEN 1 ELSE 0 END) AS costos_nulos,
    SUM(CASE WHEN costo_unitario_clp < 0 THEN 1 ELSE 0 END) AS costos_negativos
FROM detalle_pedidos;



Running query in 'sqlite:///../data/distribuidora_cordillera.sqlite'

total_lineas,cantidades_nulas,cantidades_no_positivas,precios_unitarios_nulos,precios_unitarios_negativos,costos_nulos,costos_negativos
61647,0,0,0,0,0,0



#### 1.4.4.1 Resultados de completitud y validez cuantitativa

Se analizaron 61.647 líneas de la tabla `detalle_pedidos`, evaluando la presencia de valores nulos y valores potencialmente inválidos en cantidades, precios netos y costos unitarios.

| Indicador | Registros detectados |
|---|---:|
| Total de líneas evaluadas | 61.647 |
| Cantidades nulas | 0 |
| Cantidades no positivas | 0 |
| Precios netos unitarios nulos | 0 |
| Precios netos unitarios negativos | 0 |
| Costos unitarios nulos | 0 |
| Costos unitarios negativos | 0 |

#### Hallazgos

No se detectaron valores ausentes en los tres atributos cuantitativos evaluados.

Todas las líneas presentan cantidades positivas, mientras que los precios netos y costos unitarios no registran valores negativos.

#### Interpretación

Los resultados son favorables para la construcción de indicadores comerciales, dado que no se identificaron anomalías en las condiciones examinadas.

Sin embargo, estos controles no permiten asegurar por sí solos la consistencia económica de los importes ni la ausencia de valores atípicos.

#### Conclusión

Los atributos evaluados superan los controles iniciales de completitud y validez cuantitativa.

La siguiente fase profundizará en valores iguales a cero, porcentajes de descuento y relaciones entre precios y costos.

**Estado del control:** conforme, sin anomalías detectadas en las condiciones evaluadas.



#### 1.4.4.2 Evaluación de precios, costos y descuentos

**Objetivo**

Identificar condiciones comerciales que podrían comprometer la interpretación de ingresos y márgenes brutos estimados.

**Metodología**

Se analizarán registros con precios o costos unitarios iguales a cero, descuentos fuera del rango porcentual esperado y costos superiores a los precios netos unitarios.

Estas condiciones se tratarán inicialmente como alertas de revisión, sin asumir que representan errores de registro.


In [22]:
%%sql

SELECT
    COUNT(*) AS total_lineas,
    SUM(CASE WHEN neto_unitario_clp = 0 THEN 1 ELSE 0 END) AS precios_cero,
    SUM(CASE WHEN costo_unitario_clp = 0 THEN 1 ELSE 0 END) AS costos_cero,
    SUM(CASE WHEN descuento_pct < 0 OR descuento_pct > 100 THEN 1 ELSE 0 END) AS descuentos_fuera_rango,
    SUM(CASE WHEN descuento_pct IS NULL THEN 1 ELSE 0 END) AS descuentos_nulos,
    SUM(CASE WHEN costo_unitario_clp > neto_unitario_clp THEN 1 ELSE 0 END) AS lineas_bajo_costo
FROM detalle_pedidos;

Running query in 'sqlite:///../data/distribuidora_cordillera.sqlite'

total_lineas,precios_cero,costos_cero,descuentos_fuera_rango,descuentos_nulos,lineas_bajo_costo
61647,0,0,0,0,0



#### 1.4.4.2.1 Resultados de la evaluación comercial

Se analizaron 61.647 líneas de pedidos para identificar condiciones potencialmente anómalas en los precios netos, costos unitarios y descuentos registrados.

| Indicador | Registros detectados |
|---|---:|
| Líneas evaluadas | 61.647 |
| Precios netos iguales a cero | 0 |
| Costos unitarios iguales a cero | 0 |
| Descuentos fuera del rango 0–100 % | 0 |
| Descuentos nulos | 0 |
| Líneas con costo superior al precio neto | 0 |

#### Hallazgos

No se detectaron precios netos ni costos unitarios iguales a cero.

Los descuentos informados se encuentran dentro del rango porcentual esperado y no presentan valores nulos.

Adicionalmente, ninguna línea presenta un costo unitario superior al precio neto unitario registrado.

#### Interpretación

Los resultados respaldan la consistencia inicial de los atributos comerciales evaluados.

Sin embargo, la ausencia de estas anomalías no garantiza que los precios netos correspondan correctamente a los precios originales y descuentos aplicados.

Tampoco permite determinar la rentabilidad final de las operaciones, dado que todavía no se incorporan gastos indirectos ni otros componentes financieros.

#### Conclusión

Los registros superan los controles de validez comercial aplicados.

La siguiente revisión evaluará la consistencia matemática entre precios unitarios, descuentos y precios netos.

**Estado del control:** conforme, sin anomalías detectadas.



#### 1.4.4.3 Consistencia entre precios y descuentos

**Objetivo**

Evaluar la correspondencia matemática entre los precios unitarios, los porcentajes de descuento y los importes netos registrados.

**Metodología**

Se explorará la relación entre `precio_unitario_clp`, `descuento_pct` y `neto_unitario_clp`.

Posteriormente, se contrastarán los importes netos registrados con los valores calculados a partir del precio original y el descuento informado.

Se considerarán posibles diferencias por redondeo antes de clasificar los registros como inconsistentes.


In [31]:
%%sql

SELECT
    detalle_id,
    precio_unitario_clp,
    descuento_pct,
    neto_unitario_clp,
    precio_unitario_clp * (1 - descuento_pct/100.0) AS neto_calculado,
    neto_unitario_clp - (precio_unitario_clp * (1 - descuento_pct / 100.0)) AS diferencia_neto
    FROM detalle_pedidos;

Running query in 'sqlite:///../data/distribuidora_cordillera.sqlite'

detalle_id,precio_unitario_clp,descuento_pct,neto_unitario_clp,neto_calculado,diferencia_neto
1,4750,5,4513,4512.5,0.5
2,3580,10,3222,3222.0,0.0
3,3950,10,3555,3555.0,0.0
4,1370,0,1370,1370.0,0.0
5,1490,0,1490,1490.0,0.0
6,3700,5,3515,3515.0,0.0
7,2800,15,2380,2380.0,0.0
8,5480,0,5480,5480.0,0.0
9,3040,5,2888,2888.0,0.0
10,2770,0,2770,2770.0,0.0


In [33]:
%%sql

WITH validacion_precios AS (
    SELECT
        neto_unitario_clp,
        precio_unitario_clp * (1 - descuento_pct / 100.0) AS neto_calculado,
        ROUND(precio_unitario_clp * (1 - descuento_pct / 100.0), 0) AS neto_calculado_redondeado
    FROM detalle_pedidos
)
SELECT
    COUNT(*) AS total_lineas,
    SUM(CASE WHEN ABS(neto_unitario_clp - neto_calculado) > 0.000001 THEN 1 ELSE 0 END) AS lineas_con_diferencia_sin_redondeo,
    SUM(CASE WHEN ABS(neto_unitario_clp - neto_calculado_redondeado) > 0.000001 THEN 1 ELSE 0 END) AS lineas_con_diferencia_con_redondeo,
    MAX(ABS(neto_unitario_clp - neto_calculado)) AS max_diferencia_sin_redondeo
FROM validacion_precios;

Running query in 'sqlite:///../data/distribuidora_cordillera.sqlite'

total_lineas,lineas_con_diferencia_sin_redondeo,lineas_con_diferencia_con_redondeo,max_diferencia_sin_redondeo
61647,10103,0,0.5



#### 1.4.4.3.1 Resultados de consistencia matemática

Se evaluaron las 61.647 líneas de `detalle_pedidos`, contrastando los precios netos registrados con los importes teóricos calculados mediante la aplicación del descuento porcentual.

Se realizaron dos comparaciones: una conservando la precisión decimal y otra redondeando el resultado al peso entero.

| Indicador | Resultado |
|---|---:|
| Líneas evaluadas | 61.647 |
| Líneas con diferencias sin redondeo | 10.103 |
| Líneas con diferencias después del redondeo | 0 |
| Máxima diferencia absoluta sin redondear | $0,50 |

#### Hallazgos

El 16,39 % de las líneas presenta diferencias entre el precio neto registrado y el cálculo teórico sin redondear.

La máxima diferencia absoluta detectada fue de $0,50 por unidad.

Al aplicar `ROUND(..., 0)` al importe teórico, las diferencias desaparecen en la totalidad de las líneas evaluadas.

#### Interpretación

Los resultados son consistentes con una regla de cálculo que aplica el descuento porcentual al precio unitario y redondea el resultado al peso entero.

Las diferencias detectadas antes del redondeo no constituyen evidencia de errores monetarios bajo esta regla.

#### Conclusión

La totalidad de las líneas evaluadas presenta correspondencia matemática entre precio unitario, descuento aplicado y precio neto registrado, considerando el redondeo al peso entero.

Esta validación respalda la utilización de `neto_unitario_clp` para construir indicadores comerciales.

No obstante, no verifica la exactitud comercial de los precios y descuentos originales ni la correcta determinación de los costos.

**Estado del control:** conforme, sin diferencias posteriores al redondeo.



### 1.5 Conciliación de indicadores comerciales

#### Objetivo

Verificar la consistencia de los importes netos calculados a partir de las líneas de pedidos, considerando los distintos estados comerciales.

Esta conciliación permitirá establecer valores de referencia para el futuro modelo analítico y reducir el riesgo de duplicaciones u omisiones en las operaciones JOIN.

#### Metodología

Se calculará el importe neto de cada línea mediante la multiplicación de `cantidad` por `neto_unitario_clp`.

Posteriormente, los importes serán agregados según el estado del pedido y contrastados con los totales históricos conocidos.

Se distinguirán los importes asociados a pedidos entregados, pendientes y cancelados, evitando interpretar automáticamente todos ellos como ventas efectivamente realizadas.


In [50]:
%%sql

WITH detalle_importe_pedidos AS (
    SELECT
        p.estado AS estado,
        dp.pedido_id AS pedido_id,
        COUNT(*) AS total_lineas,
        SUM(dp.neto_unitario_clp * dp.cantidad) AS importe_neto_clp
    FROM detalle_pedidos AS dp
    INNER JOIN pedidos AS p
    ON dp.pedido_id = p.pedido_id
    GROUP BY p.estado, dp.pedido_id
)
SELECT
    estado,
    COUNT(DISTINCT pedido_id) AS total_pedidos,
    SUM(total_lineas) AS total_lineas,
    SUM(importe_neto_clp) OVER(
        PARTITION BY estado
        ORDER BY importe_neto_clp DESC ) AS importe_neto_acumulado
FROM detalle_importe_pedidos
GROUP BY estado, pedido_id
;    

Running query in 'sqlite:///../data/distribuidora_cordillera.sqlite'

estado,total_pedidos,total_lineas,importe_neto_acumulado
cancelado,1,6,903684
cancelado,1,5,1763544
cancelado,1,6,2621136
cancelado,1,6,3471294
cancelado,1,6,4260528
cancelado,1,6,5018256
cancelado,1,6,5775708
cancelado,1,6,6531900
cancelado,1,6,7274994
cancelado,1,5,8015478


In [59]:
%%sql

  SELECT
        p.estado,
        COUNT(DISTINCT p.pedido_id) AS total_pedidos,
        COUNT(*) AS total_lineas,
        SUM(dp.neto_unitario_clp * dp.cantidad) AS importe_neto_clp,
        ROUND(SUM(dp.neto_unitario_clp * dp.cantidad) * 100.0 / SUM(SUM(dp.neto_unitario_clp * dp.cantidad)) OVER(), 2) AS pct_importe_neto
    FROM detalle_pedidos AS dp
    INNER JOIN pedidos AS p
    ON dp.pedido_id = p.pedido_id
    GROUP BY p.estado
    ORDER BY importe_neto_clp DESC;

Running query in 'sqlite:///../data/distribuidora_cordillera.sqlite'

estado,total_pedidos,total_lineas,importe_neto_clp,pct_importe_neto
entregado,14734,58522,4434202034,95.02
cancelado,698,2835,209351774,4.49
pendiente,75,290,23133158,0.5



#### 1.5.1 Resultados de conciliación por estado comercial

Se conciliaron los importes netos de las 61.647 líneas de detalle con los estados de sus respectivos pedidos.

El importe neto de cada línea se calculó multiplicando la cantidad por el precio neto unitario registrado.

| Estado | Pedidos | Líneas | Importe neto (CLP) | Participación |
|---|---:|---:|---:|---:|
| Entregado | 14.734 | 58.522 | $4.434.202.034 | 95,02 % |
| Cancelado | 698 | 2.835 | $209.351.774 | 4,49 % |
| Pendiente | 75 | 290 | $23.133.158 | 0,50 % |
| **Total** | **15.507** | **61.647** | **$4.666.686.966** | **100 %** |

*Los porcentajes individuales pueden presentar diferencias de suma debido al redondeo.*

#### Hallazgos

- Los conteos de pedidos y líneas coinciden con los valores históricos de referencia.
- El 95,02 % del importe neto registrado corresponde a pedidos entregados.
- El 4,98 % restante corresponde a pedidos cancelados y pendientes, considerando los porcentajes sin redondear.

#### Conclusión

La conciliación no presenta diferencias respecto de los totales de referencia.

Para los futuros indicadores comerciales se distinguirá entre el valor neto de pedidos entregados y el valor de pedidos pendientes o cancelados.

Los importes asociados a pedidos entregados no se interpretarán automáticamente como facturación emitida o ingresos efectivamente cobrados.

**Estado del control:** conforme.



### 1.5.2 Conciliación de importes por segmento comercial

#### Objetivo

Verificar que la distribución del importe neto de pedidos entregados entre los segmentos comerciales sea consistente con el total general previamente conciliado.

#### Metodología

Se relacionarán las tablas `clientes`, `pedidos` y `detalle_pedidos` para calcular indicadores agrupados por segmento comercial.

Se considerarán exclusivamente los pedidos con estado `entregado`.

Los resultados se contrastarán con el importe neto total de pedidos entregados obtenido en la sección anterior, con el fin de identificar posibles diferencias asociadas a las relaciones entre tablas o a la agregación de importes.


In [63]:
%%sql

SELECT
    c.segmento,
    COUNT(DISTINCT p.pedido_id) AS pedidos_entregados,
    COUNT(*) AS lineas_entregadas,
    SUM(dp.neto_unitario_clp * dp.cantidad) AS importe_neto_segmento_clp,
    ROUND(100.0 *SUM(dp.neto_unitario_clp * dp.cantidad) / SUM(SUM(dp.neto_unitario_clp * dp.cantidad)) OVER(), 2) AS pct_participacion_segmento
FROM clientes AS c
INNER JOIN pedidos AS p ON c.cliente_id = p.cliente_id
INNER JOIN detalle_pedidos AS dp ON p.pedido_id = dp.pedido_id
WHERE p.estado = 'entregado'
GROUP BY c.segmento
ORDER BY importe_neto_segmento_clp DESC;

Running query in 'sqlite:///../data/distribuidora_cordillera.sqlite'

segmento,pedidos_entregados,lineas_entregadas,importe_neto_segmento_clp,pct_participacion_segmento
Almacen,8295,32925,2653879596,59.85
Minimarket,3666,14492,1154642886,26.04
Horeca,2773,11105,625679552,14.11



#### 1.5.2.1 Resultados de conciliación por segmento

Se evaluó la distribución del importe neto correspondiente a pedidos entregados entre los tres segmentos comerciales de Distribuidora Cordillera.

El análisis relacionó las tablas `clientes`, `pedidos` y `detalle_pedidos`, considerando la granularidad de las líneas de pedido y contabilizando los pedidos mediante sus identificadores distintos.

| Segmento | Pedidos entregados | Líneas | Importe neto (CLP) | Participación |
|---|---:|---:|---:|---:|
| Almacen | 8.295 | 32.925 | $2.653.879.596 | 59,85 % |
| Minimarket | 3.666 | 14.492 | $1.154.642.886 | 26,04 % |
| Horeca | 2.773 | 11.105 | $625.679.552 | 14,11 % |
| **Total** | **14.734** | **58.522** | **$4.434.202.034** | **100 %** |

#### Hallazgos

- El segmento Almacen concentra el 59,85 % del importe neto de pedidos entregados, seguido por Minimarket (26,04 %) y Horeca (14,11 %).
- Los 14.734 pedidos entregados contienen 58.522 líneas de detalle.
- Los importes netos segmentados coinciden con el total general obtenido en la conciliación por estado comercial.
- No se identificaron diferencias de conciliación entre las agregaciones evaluadas.

#### Consideraciones técnicas

La relación entre pedidos y líneas de detalle es de uno a muchos. Por esta razón, el conteo de pedidos requiere evitar duplicaciones al trabajar con información de detalle.

La participación porcentual corresponde a la proporción del importe neto entregado de cada segmento respecto del importe neto total entregado.

No representa un porcentaje acumulado ni una medida de rentabilidad.

#### Conclusión

La conciliación confirma la consistencia de los importes netos registrados para pedidos entregados al agregarlos por segmento comercial.

Los resultados establecen una línea base para la construcción de indicadores comerciales y posteriores análisis de participación, concentración y márgenes brutos estimados.

Los importes de pedidos entregados no se consideran automáticamente equivalentes a facturación emitida o ingresos efectivamente cobrados.

**Estado del control:** conforme, sin diferencias detectadas.



## 1.6 Síntesis de auditoría y recomendaciones

### Resumen ejecutivo

Se realizó una auditoría exploratoria de la base de datos ficticia de Distribuidora Cordillera, orientada a evaluar la confiabilidad de la información utilizada en futuros análisis de Business Intelligence.

La base contiene 17 tablas. La revisión inicial se concentró en cuatro entidades principales: `clientes`, `pedidos`, `detalle_pedidos` y `productos`.

Se evaluaron controles de unicidad, integridad referencial, completitud, consistencia temporal, validez de atributos comerciales y conciliación de importes.

Los resultados fueron favorables dentro del alcance definido, sin detectarse inconsistencias en los controles aplicados.

### 1.6.1 Principales resultados

| Dimensión | Resultado | Evaluación |
|---|---|---|
| Inventario de datos | 17 tablas identificadas | Documentado |
| Unicidad de identificadores | Sin duplicados ni identificadores nulos en las cuatro tablas seleccionadas | Conforme |
| Integridad referencial | 0 registros huérfanos en las tres relaciones evaluadas | Conforme |
| Completitud de pedidos | 773 registros sin fechas de entrega y vencimiento | Patrón explicado por estado |
| Consistencia temporal | 0 anomalías en los controles aplicados a 14.734 pedidos entregados | Conforme |
| Cantidades, precios y costos | Sin valores nulos o fuera de los rangos evaluados | Conforme |
| Consistencia de descuentos | 0 diferencias después del redondeo al peso | Conforme |
| Conciliación monetaria | $4.666.686.966 CLP considerando todos los estados | Conforme |
| Importe neto entregado | $4.434.202.034 CLP | Conforme |

### 1.6.2 Hallazgos relevantes

**Completitud condicionada por el estado comercial**

Se identificaron 773 pedidos sin fechas de entrega y vencimiento, correspondientes exclusivamente a pedidos pendientes y cancelados.

La ausencia de estos atributos responde a un patrón consistente con los estados registrados. Sin embargo, la interpretación operativa de `fecha_vencimiento` requiere conocer las reglas del sistema.

**Consistencia matemática de precios**

Se identificaron 10.103 líneas con diferencias entre el precio neto registrado y el cálculo teórico sin redondear.

La máxima diferencia absoluta fue de $0,50 por unidad. Al aplicar redondeo al peso entero, las diferencias desaparecieron en las 61.647 líneas evaluadas.

**Conciliación de importes**

Los importes obtenidos mediante agregaciones por estado y segmento coincidieron con los totales de referencia.

La conciliación permitió verificar la correcta agregación de importes a partir de las líneas de detalle y la necesidad de distinguir el conteo de pedidos del conteo de líneas.

### 1.6.3 Limitaciones del análisis

La auditoría se concentró en cuatro de las 17 tablas disponibles y no constituye una evaluación integral de todas las entidades del sistema.

Los controles aplicados permiten detectar determinadas anomalías, pero no garantizan la exactitud comercial de cada registro ni la ausencia de errores no contemplados.

Asimismo, los importes netos asociados a pedidos entregados no representan necesariamente facturación emitida ni ingresos efectivamente cobrados.

La rentabilidad final tampoco puede determinarse exclusivamente mediante precios y costos unitarios, debido a la existencia de gastos operativos, logísticos, financieros y otros componentes no considerados.

### 1.6.4 Recomendaciones para el modelo analítico

1. Mantener controles reproducibles de unicidad, integridad referencial y conciliación de importes.
2. Definir explícitamente la granularidad de cada tabla analítica para prevenir duplicaciones en las agregaciones.
3. Diferenciar los importes de pedidos entregados, pendientes y cancelados en los indicadores comerciales.
4. Utilizar el precio neto unitario registrado, cuya correspondencia matemática con los descuentos fue validada bajo redondeo al peso.
5. Documentar las definiciones de indicadores y las reglas de negocio utilizadas.
6. Ampliar progresivamente las validaciones hacia inventario, compras, devoluciones y otras entidades relevantes.

### 1.6.5 Conclusión general

La información examinada presenta condiciones favorables para avanzar hacia la construcción de un modelo analítico comercial.

Los controles de calidad e integridad aplicados no detectaron inconsistencias que impidan utilizar las cuatro tablas seleccionadas dentro del alcance evaluado.

Los resultados establecen una línea base verificable para el desarrollo de indicadores comerciales, análisis de clientes, participación por segmento y márgenes brutos estimados.

**Próxima etapa:** diseño del modelo analítico SQL, definición de granularidad, construcción de vistas reutilizables y preparación de fuentes para herramientas de Business Intelligence.

---

**Nota metodológica:** el proyecto utiliza una base de datos ficticia y tiene fines de aprendizaje aplicado y demostración de competencias técnicas.
